In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.util
import json
import math
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone

PROTOCOL_ID = "TRKH_PRETRAINED_CLASSF_B2_TEMPERED_P05_20260731"
EXPERIMENT_KEY = "b2-tempered-p05"
KAGGLE_NOTEBOOK_CONTRACT = "TRKH_KAGGLE_B2_T4_OFFLINE_V3_20260801"
KAGGLE_RUNTIME_RELEASE = "v170"
KAGGLE_RUNTIME_SOURCE = "https://github.com/Kaggle/docker-python/releases/tag/v170-GPU-bdf9e0538555f90453619adefb49ba40cfa136db44a9c9be7a42ea715c0aa068"
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
RUN_TAG = "kaggle_b2_tempered_p05_v3"
AUTO_RESUME = False
CONFIRM_FULL = False  # Chi doi True sau khi doc metrics probe.
KAGGLE_COMPACT_PROGRESS = True  # Tat tqdm dong; van giu metric log moi epoch.
if KAGGLE_COMPACT_PROGRESS:
    os.environ["TQDM_DISABLE"] = "1"
else:
    os.environ.pop("TQDM_DISABLE", None)
RUN_PROBE = not AUTO_RESUME
PROBE_MIN_MACRO_F1 = 0.55
PROBE_MIN_CLASS1_F1 = 0.30
DATA_YAML_OVERRIDE = ""  # De trong neu chi co mot data.yaml tuong thich.
DATA_ROOT_OVERRIDE = ""  # De trong neu train/val nam canh data.yaml.
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
EXPECTED_CLASS_NAMES = (
    "Xoai_Song_Chua_KhoDap",
    "Xoai_Song_ChuaNhe_CoNguyCo",
    "Xoai_Chin_NgotThanh_DeDap",
    "Xoai_ChinGia_NgotGat_KhongVanChuyen",
    "Xoai_Hu_KhongAnDuoc",
)
EXPECTED_SOURCE_COMMIT = "7f7f0883cbb71b6a5620fee86c15b996c400a813"
EXPECTED_SOURCE_TREE_SHA256 = "7c8752efe6acb728ed913abe5db165b218c94177e3ef13f5b37ce1c62b19d965"
DINO_SHA256 = "2a1ec16ae28ffa07bc0ead0241ee7df9fc26451fe6f9f839b7b3afa0a906b040"
DINO_BYTES = 86362376
EXPECTED_VENDORED_TIMM_TREE_SHA256 = "c3f21c6f3ef4ac466d46e33494c726e4d80ce3329e73b537f68a47fe066dc856"
EXPECTED_DINOV3_LICENSE_SHA256 = "25d122eb8f5b880fd23c736fb6ea8018ee45c12237e00b8a86d14c653904999e"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print(PROTOCOL_ID, "auto_resume=", AUTO_RESUME, "run_probe=", RUN_PROBE, "compact_progress=", KAGGLE_COMPACT_PROGRESS)


In [ ]:
import stat
import zipfile

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

ASSET_ZIP_GLOB = "TRKH_KAGGLE_B2_T4_UPLOAD_BUNDLE_*.zip"
DATASET_ZIP_GLOB = "TRKH_CLASSF_DEV_DATASET_*.zip"
ASSET_MANIFEST_NAME = "TRKH_KAGGLE_B2_UPLOAD_MANIFEST.json"
EXTRACT_MARKER_NAME = ".trkh_safe_extract.json"
MAX_ZIP_MEMBERS = 200_000
MAX_ZIP_MEMBER_BYTES = 4 * 1024 ** 3
MAX_ZIP_TOTAL_BYTES = 16 * 1024 ** 3

def discover_optional_zip(pattern, label):
    candidates = sorted({path.resolve() for path in INPUT_ROOT.rglob(pattern) if path.is_file()})
    if len(candidates) > 1:
        raise RuntimeError(f"Chi duoc attach toi da 1 {label} ZIP, tim thay: {candidates}")
    return candidates[0] if candidates else None

def safe_member_target(root, member_name):
    normalized = str(member_name).replace("\\", "/")
    if not normalized or "\x00" in normalized or normalized.startswith("/") or re.match(r"^[A-Za-z]:", normalized):
        raise RuntimeError(f"Unsafe ZIP member path: {member_name!r}")
    parts = [part for part in normalized.split("/") if part not in ("", ".")]
    if not parts or any(part == ".." for part in parts):
        raise RuntimeError(f"Unsafe ZIP traversal path: {member_name!r}")
    root = root.resolve()
    target = (root / Path(*parts)).resolve()
    try:
        target.relative_to(root)
    except ValueError as error:
        raise RuntimeError(f"ZIP member escapes extraction root: {member_name!r}") from error
    return target, tuple(parts)

def safe_extract_zip(zip_path, label, forbidden_components=()):
    if zip_path is None:
        return None, None
    zip_digest = sha256(zip_path)
    extract_root = WORK_ROOT / f"_trkh_{label}_{zip_digest[:16]}"
    marker_path = extract_root / EXTRACT_MARKER_NAME
    forbidden = {str(value).casefold() for value in forbidden_components}
    validated = []
    seen_paths = set()
    total_bytes = 0
    with zipfile.ZipFile(zip_path, "r") as archive:
        infos = archive.infolist()
        if not infos or len(infos) > MAX_ZIP_MEMBERS:
            raise RuntimeError(f"ZIP member count khong hop le: {len(infos)}")
        for info in infos:
            if info.flag_bits & 0x1:
                raise RuntimeError(f"Khong chap nhan encrypted ZIP member: {info.filename}")
            target, parts = safe_member_target(extract_root, info.filename)
            if len(parts) == 1 and parts[0].casefold() == EXTRACT_MARKER_NAME.casefold():
                raise RuntimeError(f"ZIP member uses reserved extraction marker: {info.filename}")
            if any(part.casefold() in forbidden for part in parts):
                raise RuntimeError(f"ZIP {label} chua thanh phan bi cam: {info.filename}")
            normalized_key = "/".join(parts).casefold()
            if normalized_key in seen_paths:
                raise RuntimeError(f"Duplicate ZIP member path: {info.filename}")
            seen_paths.add(normalized_key)
            mode = (int(info.external_attr) >> 16) & 0xFFFF
            file_type = stat.S_IFMT(mode)
            is_directory = bool(info.is_dir() or (file_type and stat.S_ISDIR(mode)))
            if stat.S_ISLNK(mode) or (file_type not in (0, stat.S_IFREG, stat.S_IFDIR)):
                raise RuntimeError(f"Khong chap nhan symlink/special ZIP member: {info.filename}")
            if info.file_size < 0 or info.file_size > MAX_ZIP_MEMBER_BYTES:
                raise RuntimeError(f"ZIP member qua lon: {info.filename} ({info.file_size})")
            if not is_directory:
                total_bytes += int(info.file_size)
            validated.append((info, target, is_directory))
        if total_bytes > MAX_ZIP_TOTAL_BYTES:
            raise RuntimeError(f"ZIP uncompressed payload qua lon: {total_bytes}")
        regular_keys = {
            target.relative_to(extract_root).as_posix().casefold()
            for _, target, is_directory in validated
            if not is_directory
        }
        for key in regular_keys:
            parts = key.split("/")
            if any("/".join(parts[:index]) in regular_keys for index in range(1, len(parts))):
                raise RuntimeError(f"ZIP regular file shadows a parent directory: {key}")
        if extract_root.exists():
            if not marker_path.is_file():
                raise RuntimeError(f"Thu muc extract dang do, hay restart Kaggle session: {extract_root}")
            marker = json.loads(marker_path.read_text(encoding="utf-8"))
            expected_marker = {
                "status": "passed",
                "label": label,
                "zip_sha256": zip_digest,
                "member_count": len(validated),
                "regular_file_count": len(regular_keys),
                "uncompressed_regular_bytes": total_bytes,
            }
            marker_mismatches = {
                key: {"observed": marker.get(key), "expected": expected}
                for key, expected in expected_marker.items()
                if marker.get(key) != expected
            }
            if marker_mismatches:
                raise RuntimeError(f"Extraction marker mismatch: {marker_mismatches}")
            missing_or_resized = [
                str(target)
                for info, target, is_directory in validated
                if (is_directory and not target.is_dir())
                or (not is_directory and (not target.is_file() or target.stat().st_size != info.file_size))
            ]
            if missing_or_resized:
                raise RuntimeError(f"Extracted ZIP tree changed: {missing_or_resized[:20]}")
            actual_regular_keys = {
                path.relative_to(extract_root).as_posix().casefold()
                for path in extract_root.rglob("*")
                if path.is_file() and path.resolve() != marker_path.resolve()
            }
            if actual_regular_keys != regular_keys:
                raise RuntimeError(
                    "Extracted ZIP missing/extra files: "
                    f"missing={sorted(regular_keys - actual_regular_keys)}, "
                    f"extra={sorted(actual_regular_keys - regular_keys)}"
                )
            return extract_root, marker
        free_bytes = shutil.disk_usage(WORK_ROOT).free
        if total_bytes > max(0, free_bytes - 1024 ** 3):
            raise RuntimeError(f"Khong du working disk: need={total_bytes}, free={free_bytes}")
        extract_root.mkdir(parents=True, exist_ok=False)
        for info, target, is_directory in validated:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info, "r") as source, target.open("xb") as destination:
                shutil.copyfileobj(source, destination, length=1024 * 1024)
    marker = {
        "schema_version": 1,
        "status": "passed",
        "label": label,
        "zip_path": str(zip_path),
        "zip_sha256": zip_digest,
        "member_count": len(validated),
        "regular_file_count": sum(not is_directory for _, _, is_directory in validated),
        "uncompressed_regular_bytes": total_bytes,
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return extract_root, marker

def discover_expanded_asset_root():
    manifests = sorted({
        path.resolve()
        for path in INPUT_ROOT.rglob(ASSET_MANIFEST_NAME)
        if path.is_file()
    })
    if len(manifests) > 1:
        raise RuntimeError(f"Can dung toi da 1 expanded asset manifest, tim thay: {manifests}")
    return manifests[0].parent if manifests else None

def kaggle_input_mount(path):
    root = INPUT_ROOT.resolve()
    resolved = Path(path).resolve()
    try:
        relative = resolved.relative_to(root)
    except ValueError as error:
        raise RuntimeError(f"Input path nam ngoai /kaggle/input: {resolved}") from error
    if not relative.parts:
        raise RuntimeError(f"Khong xac dinh duoc Kaggle Dataset mount cho: {resolved}")
    return (root / relative.parts[0]).resolve()

ASSET_ZIP = discover_optional_zip(ASSET_ZIP_GLOB, "asset bundle")
ASSET_EXPANDED_ROOT = discover_expanded_asset_root()
if ASSET_ZIP is not None and ASSET_EXPANDED_ROOT is not None:
    raise RuntimeError(
        "Asset input mo ho: dong thoi co ZIP va expanded manifest. "
        "Chi attach mot representation."
    )
DATASET_ZIP = discover_optional_zip(DATASET_ZIP_GLOB, "dataset")
if ASSET_ZIP is not None:
    ASSET_EXTRACT_ROOT, ASSET_ZIP_CONTRACT = safe_extract_zip(ASSET_ZIP, "assets")
    ASSET_ZIP_CONTRACT = {
        **ASSET_ZIP_CONTRACT,
        "input_mode": "archive_file",
    }
    ASSET_INPUT_ANCHOR = ASSET_ZIP
elif ASSET_EXPANDED_ROOT is not None:
    ASSET_EXTRACT_ROOT = ASSET_EXPANDED_ROOT.resolve()
    ASSET_ZIP_CONTRACT = {
        "schema_version": 1,
        "status": "resolved_pending_manifest",
        "label": "assets",
        "input_mode": "kaggle_mounted_expanded",
        "root": str(ASSET_EXTRACT_ROOT),
    }
    ASSET_INPUT_ANCHOR = ASSET_EXTRACT_ROOT / ASSET_MANIFEST_NAME
else:
    raise RuntimeError(
        f"Attach exactly one asset input: ZIP matching {ASSET_ZIP_GLOB} "
        f"or an expanded tree containing {ASSET_MANIFEST_NAME}"
    )
DATASET_EXTRACT_ROOT, DATASET_ZIP_CONTRACT = safe_extract_zip(
    DATASET_ZIP,
    "dataset",
    forbidden_components={"test"},
)
if DATASET_ZIP_CONTRACT is not None:
    DATASET_ZIP_CONTRACT = {
        **DATASET_ZIP_CONTRACT,
        "input_mode": "archive_file",
    }
ASSET_INPUT_MOUNT = kaggle_input_mount(ASSET_INPUT_ANCHOR)
SEARCH_ROOTS = [INPUT_ROOT]
for extracted_root in (ASSET_EXTRACT_ROOT, DATASET_EXTRACT_ROOT):
    if extracted_root is not None:
        SEARCH_ROOTS.append(extracted_root)

def find_in_search_roots(pattern):
    return sorted({path.resolve() for root in SEARCH_ROOTS for path in root.rglob(pattern) if path.is_file()})

if ASSET_EXTRACT_ROOT is not None:
    manifest_path = ASSET_EXTRACT_ROOT / ASSET_MANIFEST_NAME
    if not manifest_path.is_file():
        raise RuntimeError(f"Asset ZIP thieu root manifest: {manifest_path}")
    asset_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if not isinstance(asset_manifest, dict) or not isinstance(asset_manifest.get("files"), list):
        raise RuntimeError("Asset manifest phai la object voi files inventory")
    required_manifest_values = {
        "notebook_contract": KAGGLE_NOTEBOOK_CONTRACT,
        "protocol": PROTOCOL_ID,
        "experiment": EXPERIMENT_KEY,
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "source_tree_sha256": EXPECTED_SOURCE_TREE_SHA256,
        "dino_sha256": DINO_SHA256,
        "vendored_timm_version": "1.0.27",
        "vendored_timm_tree_sha256": EXPECTED_VENDORED_TIMM_TREE_SHA256,
        "dinov3_license_sha256": EXPECTED_DINOV3_LICENSE_SHA256,
        "kaggle_runtime_target": "v170 GPU / Python 3.12 / Torch 2.10 cu128 / T4",
    }
    manifest_mismatches = {
        key: {"observed": asset_manifest.get(key), "expected": expected}
        for key, expected in required_manifest_values.items()
        if str(asset_manifest.get(key, "")).lower() != expected.lower()
    }
    if manifest_mismatches:
        raise RuntimeError(f"Asset manifest provenance mismatch: {manifest_mismatches}")
    declared_files = {}
    for entry in asset_manifest["files"]:
        if not isinstance(entry, dict):
            raise RuntimeError(f"Invalid asset manifest file entry: {entry!r}")
        target, parts = safe_member_target(ASSET_EXTRACT_ROOT, entry.get("path", ""))
        relative = "/".join(parts)
        key = relative.casefold()
        if key in declared_files:
            raise RuntimeError(f"Duplicate asset manifest path: {relative}")
        declared_files[key] = {
            "path": relative,
            "target": target,
            "size_bytes": int(entry.get("size_bytes", -1)),
            "sha256": str(entry.get("sha256", "")).lower(),
        }
    invalid_asset_entries = [
        str(path)
        for path in ASSET_EXTRACT_ROOT.rglob("*")
        if path.is_symlink() or not (path.is_file() or path.is_dir())
    ]
    if invalid_asset_entries:
        raise RuntimeError(
            f"Asset input chua symlink/special entries: {invalid_asset_entries[:20]}"
        )
    actual_files = {
        path.relative_to(ASSET_EXTRACT_ROOT).as_posix().casefold(): path
        for path in ASSET_EXTRACT_ROOT.rglob("*")
        if path.is_file()
        and path.resolve() not in {manifest_path.resolve(), (ASSET_EXTRACT_ROOT / EXTRACT_MARKER_NAME).resolve()}
    }
    if set(declared_files) != set(actual_files):
        raise RuntimeError(
            "Asset manifest missing/extra files: "
            f"missing={sorted(set(actual_files) - set(declared_files))}, "
            f"extra={sorted(set(declared_files) - set(actual_files))}"
        )
    for key, declared in declared_files.items():
        path = actual_files[key]
        if declared["path"] != path.relative_to(ASSET_EXTRACT_ROOT).as_posix():
            raise RuntimeError(f"Asset manifest path case/spelling mismatch: {declared['path']}")
        if declared["size_bytes"] != path.stat().st_size or not re.fullmatch(r"[0-9a-f]{64}", declared["sha256"]):
            raise RuntimeError(f"Asset manifest size/hash schema mismatch: {declared['path']}")
        observed_hash = sha256(path)
        if observed_hash != declared["sha256"]:
            raise RuntimeError(f"Asset digest mismatch: {declared['path']}")
    required_asset_paths = [
        ASSET_EXTRACT_ROOT / "TRKH_CLASSF_BEST_KAGGLE.ipynb",
        ASSET_EXTRACT_ROOT / "vendor" / "python" / "timm" / "__init__.py",
        ASSET_EXTRACT_ROOT / "third_party" / "dinov3" / "LICENSE.md",
        ASSET_EXTRACT_ROOT / "third_party" / "dinov3" / "PROVENANCE.json",
        ASSET_EXTRACT_ROOT / "TRKH_KAGGLE_RUNTIME_TARGET.json",
        ASSET_EXTRACT_ROOT / "TRKH_pretrained" / "SOURCE_COMMIT.txt",
        ASSET_EXTRACT_ROOT / "TRKH_pretrained" / "trkh" / "recipes" / "pretrained_classf_b0.py",
        ASSET_EXTRACT_ROOT / "weights" / "model.safetensors",
    ]
    missing_required_assets = [str(path) for path in required_asset_paths if not path.is_file()]
    if missing_required_assets:
        raise RuntimeError(f"Asset bundle thieu file bat buoc: {missing_required_assets}")

    runtime_target_path = ASSET_EXTRACT_ROOT / "TRKH_KAGGLE_RUNTIME_TARGET.json"
    provenance_path = ASSET_EXTRACT_ROOT / "third_party" / "dinov3" / "PROVENANCE.json"
    license_path = ASSET_EXTRACT_ROOT / "third_party" / "dinov3" / "LICENSE.md"
    RUNTIME_TARGET = json.loads(runtime_target_path.read_text(encoding="utf-8"))
    DINO_PROVENANCE = json.loads(provenance_path.read_text(encoding="utf-8"))
    runtime_target_required = {
        "contract": KAGGLE_NOTEBOOK_CONTRACT,
    }
    target_required = {
        "official_gpu_release": KAGGLE_RUNTIME_RELEASE,
        "python_minor": "3.12",
        "torch_family": "2.10.x+cu128",
        "torchvision_family": "0.25.x+cu128",
        "preferred_accelerator": "NvidiaTeslaT4",
        "minimum_compute_capability": [7, 5],
        "p100_supported": False,
        "fp16_grad_scaler_init_scale": 1024.0,
    }
    target_mismatches = {
        key: {"observed": RUNTIME_TARGET.get(key), "expected": expected}
        for key, expected in runtime_target_required.items()
        if RUNTIME_TARGET.get(key) != expected
    }
    observed_target = RUNTIME_TARGET.get("target", {})
    target_mismatches.update({
        f"target.{key}": {"observed": observed_target.get(key), "expected": expected}
        for key, expected in target_required.items()
        if observed_target.get(key) != expected
    })
    dependency_policy = RUNTIME_TARGET.get("dependency_policy", {})
    if dependency_policy.get("mutates_global_environment") is not False:
        target_mismatches["dependency_policy.mutates_global_environment"] = dependency_policy.get("mutates_global_environment")
    if dependency_policy.get("pip_install") is not False:
        target_mismatches["dependency_policy.pip_install"] = dependency_policy.get("pip_install")
    if dependency_policy.get("vendored_timm") != "1.0.27":
        target_mismatches["dependency_policy.vendored_timm"] = dependency_policy.get("vendored_timm")
    if target_mismatches:
        raise RuntimeError(f"Kaggle runtime target mismatch: {target_mismatches}")

    provenance_required = {
        "revision": "3bf4720a82ec2066db88137180ff1f83a675cef0",
        "weight_path": "weights/model.safetensors",
        "weight_size_bytes": DINO_BYTES,
        "weight_sha256": DINO_SHA256,
        "license_path": "third_party/dinov3/LICENSE.md",
        "license_sha256": EXPECTED_DINOV3_LICENSE_SHA256,
    }
    provenance_mismatches = {
        key: {"observed": DINO_PROVENANCE.get(key), "expected": expected}
        for key, expected in provenance_required.items()
        if DINO_PROVENANCE.get(key) != expected
    }
    if sha256(license_path) != EXPECTED_DINOV3_LICENSE_SHA256:
        provenance_mismatches["license_file_sha256"] = sha256(license_path)
    if provenance_mismatches:
        raise RuntimeError(f"DINO provenance/license mismatch: {provenance_mismatches}")

    vendored_timm_digest = hashlib.sha256()
    vendor_root = ASSET_EXTRACT_ROOT / "vendor" / "python"
    vendor_files = [item for item in vendor_root.rglob("*") if item.is_file()]
    for path in sorted(
        vendor_files,
        key=lambda item: item.relative_to(ASSET_EXTRACT_ROOT).as_posix(),
    ):
        relative = path.relative_to(ASSET_EXTRACT_ROOT).as_posix()
        vendored_timm_digest.update(relative.encode("utf-8") + b"\0")
        vendored_timm_digest.update(path.read_bytes())
    observed_timm_tree_sha256 = vendored_timm_digest.hexdigest()
    if observed_timm_tree_sha256 != EXPECTED_VENDORED_TIMM_TREE_SHA256:
        raise RuntimeError(
            "Vendored timm tree digest mismatch: "
            f"{observed_timm_tree_sha256} != {EXPECTED_VENDORED_TIMM_TREE_SHA256}"
        )
    ASSET_ZIP_CONTRACT = {
        **ASSET_ZIP_CONTRACT,
        "status": "passed",
        "manifest_path": str(manifest_path),
        "manifest_sha256": sha256(manifest_path),
        "manifest_files": len(declared_files),
    }
    ASSET_INPUT_CONTRACT = ASSET_ZIP_CONTRACT
else:
    asset_manifest = None
    manifest_path = None

repo_roots = sorted({path.parents[2] for path in find_in_search_roots("pretrained_classf_b0.py") if path.as_posix().endswith("/trkh/recipes/pretrained_classf_b0.py")})
if len(repo_roots) != 1:
    raise RuntimeError(f"Can dung 1 source TRKH_pretrained duy nhat, tim thay: {repo_roots}")
REPO_ROOT = repo_roots[0]



In [ ]:
# Runtime contract: assets are already extracted and hash-verified above.
# Do not pip-install/downgrade Kaggle's live CUDA environment.
from importlib.metadata import PackageNotFoundError, version
import io
import platform
import sysconfig

if ASSET_EXTRACT_ROOT is None:
    raise RuntimeError("B2 V3 requires one manifest-covered asset input")
if platform.system() != "Linux":
    raise RuntimeError(f"Kaggle v170 contract requires Linux, observed {platform.system()}")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "Kaggle v170 contract requires Python 3.12; "
        f"observed {sys.version.split()[0]}. Select a current Kaggle GPU image."
    )
if not INPUT_ROOT.is_dir() or not WORK_ROOT.is_dir():
    raise RuntimeError(f"Missing Kaggle roots: input={INPUT_ROOT}, working={WORK_ROOT}")

VENDORED_TIMM_ROOT = (ASSET_EXTRACT_ROOT / "vendor" / "python").resolve()
VENDORED_TIMM_PACKAGE = VENDORED_TIMM_ROOT / "timm"
if not (VENDORED_TIMM_PACKAGE / "__init__.py").is_file():
    raise RuntimeError(f"Asset ZIP is missing vendored timm 1.0.27: {VENDORED_TIMM_PACKAGE}")
sys.path.insert(0, str(VENDORED_TIMM_ROOT))

required_modules = {
    "numpy": "numpy",
    "PIL": "Pillow",
    "yaml": "PyYAML",
    "matplotlib": "matplotlib",
    "pytest": "pytest",
    "safetensors": "safetensors",
    "torch": "torch",
    "torchvision": "torchvision",
    "tqdm": "tqdm",
}
missing_modules = [
    f"{distribution} ({module})"
    for module, distribution in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if missing_modules:
    raise RuntimeError(
        "Kaggle v170 base image is missing required modules; do not mutate torch/CUDA in-place: "
        + ", ".join(missing_modules)
    )

import numpy as np
import PIL
import matplotlib
import pytest
import safetensors
import torch
import torchvision
from torchvision.ops import nms
import yaml
import timm
import tqdm as tqdm_module
from tqdm import tqdm as tqdm_progress

observed_timm_root = Path(timm.__file__).resolve()
if not observed_timm_root.is_relative_to(VENDORED_TIMM_ROOT):
    raise RuntimeError(f"Imported non-vendored timm: {observed_timm_root}")
if str(getattr(timm, "__version__", "")) != "1.0.27":
    raise RuntimeError(f"Vendored timm version drift: {getattr(timm, '__version__', None)!r}")
_progress_sink = io.StringIO()
_progress_probe = tqdm_progress(total=0, file=_progress_sink)
if KAGGLE_COMPACT_PROGRESS and not bool(_progress_probe.disable):
    raise RuntimeError("TQDM_DISABLE=1 was not honored by the Kaggle tqdm runtime")
_progress_probe.close()
if KAGGLE_COMPACT_PROGRESS and _progress_sink.getvalue():
    raise RuntimeError("Compact progress probe unexpectedly emitted a dynamic bar")
runtime_target = RUNTIME_TARGET["target"]
if not str(torch.__version__).startswith("2.10."):
    raise RuntimeError(f"Kaggle v170 torch family drift: {torch.__version__!r}")
if not str(torchvision.__version__).startswith("0.25."):
    raise RuntimeError(f"Kaggle v170 torchvision family drift: {torchvision.__version__!r}")
if str(torch.version.cuda) != "12.8":
    raise RuntimeError(f"Kaggle v170 CUDA family drift: {torch.version.cuda!r}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable the T4 GPU accelerator in Kaggle Settings")
if torch.cuda.device_count() < 1:
    raise RuntimeError("CUDA reported available but exposed no device")

gpu_inventory = []
for device_index in range(torch.cuda.device_count()):
    device_properties = torch.cuda.get_device_properties(device_index)
    gpu_inventory.append({
        "index": device_index,
        "name": device_properties.name,
        "compute_capability": [device_properties.major, device_properties.minor],
        "total_memory_bytes": int(device_properties.total_memory),
        "multi_processor_count": int(device_properties.multi_processor_count),
    })
primary_gpu = gpu_inventory[0]
if "T4" not in primary_gpu["name"].upper():
    raise RuntimeError(
        "This release is validated only for Kaggle NVIDIA T4; "
        f"observed {primary_gpu['name']!r}. Select the T4 accelerator."
    )
if tuple(primary_gpu["compute_capability"]) < (7, 5):
    raise RuntimeError(
        "T4 FP16 contract requires compute capability >= 7.5; "
        f"observed {primary_gpu['compute_capability']}"
    )

# Exercise a real FP16 CUDA matmul, synchronize, and compare with FP32 CPU.
torch.manual_seed(170)
cuda_generator = torch.Generator(device="cuda")
cuda_generator.manual_seed(170)
left = torch.randn((257, 193), device="cuda", dtype=torch.float16, generator=cuda_generator)
right = torch.randn((193, 131), device="cuda", dtype=torch.float16, generator=cuda_generator)
product_fp16 = left @ right
torch.cuda.synchronize(0)
if product_fp16.device.type != "cuda" or product_fp16.dtype != torch.float16:
    raise RuntimeError("FP16 CUDA matmul did not return a CUDA float16 tensor")
if not bool(torch.isfinite(product_fp16).all().item()):
    raise RuntimeError("FP16 CUDA matmul produced non-finite values")
reference_fp32 = left.float().cpu() @ right.float().cpu()
max_abs_error = float((product_fp16.float().cpu() - reference_fp32).abs().max().item())
reference_scale = max(1.0, float(reference_fp32.abs().max().item()))
relative_max_error = max_abs_error / reference_scale
if relative_max_error > 0.01:
    raise RuntimeError(f"FP16 CUDA matmul parity failed: relative_max_error={relative_max_error}")

# Exercise the torchvision C++/CUDA operator that commonly exposes torch ABI mismatches.
nms_boxes = torch.tensor(
    [[0.0, 0.0, 10.0, 10.0], [1.0, 1.0, 9.0, 9.0], [20.0, 20.0, 30.0, 30.0]],
    device="cuda",
)
nms_scores = torch.tensor([0.9, 0.8, 0.7], device="cuda")
nms_keep = nms(nms_boxes, nms_scores, 0.5)
torch.cuda.synchronize(0)
if nms_keep.cpu().tolist() != [0, 2]:
    raise RuntimeError(f"torchvision CUDA NMS smoke failed: {nms_keep.cpu().tolist()}")

def distribution_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

runtime_distributions = {
    name: distribution_version(name)
    for name in (
        "torch", "torchvision", "numpy", "Pillow", "PyYAML", "matplotlib",
        "pytest", "safetensors", "tqdm", "scikit-learn", "pandas", "onnx", "onnxruntime",
    )
}
runtime_modules = {
    "torch": str(Path(torch.__file__).resolve()),
    "torchvision": str(Path(torchvision.__file__).resolve()),
    "timm": str(observed_timm_root),
    "numpy": str(Path(np.__file__).resolve()),
    "Pillow": str(Path(PIL.__file__).resolve()),
    "PyYAML": str(Path(yaml.__file__).resolve()),
    "matplotlib": str(Path(matplotlib.__file__).resolve()),
    "pytest": str(Path(pytest.__file__).resolve()),
    "safetensors": str(Path(safetensors.__file__).resolve()),
    "tqdm": str(Path(tqdm_module.__file__).resolve()),
}
RUNTIME_CONTRACT = {
    "schema_version": 2,
    "status": "passed",
    "notebook_contract": KAGGLE_NOTEBOOK_CONTRACT,
    "kaggle_runtime_release_reference": KAGGLE_RUNTIME_RELEASE,
    "kaggle_runtime_source": KAGGLE_RUNTIME_SOURCE,
    "asset_runtime_target": RUNTIME_TARGET,
    "python_required": "3.12.x",
    "python": sys.version,
    "python_cache_tag": sys.implementation.cache_tag,
    "python_soabi": sysconfig.get_config_var("SOABI"),
    "platform": platform.platform(),
    "libc": list(platform.libc_ver()),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "cuda_arch_list": list(torch.cuda.get_arch_list()),
    "visible_gpu_count": torch.cuda.device_count(),
    "gpus": gpu_inventory,
    "single_gpu_training_device": 0,
    "amp_dtype_required": "fp16",
    "dataloader_worker_policy": {
        "train_workers": 2,
        "eval_workers": 2,
        "manual_safe_fallback": 0,
        "automatic_retry_after_partial_run": False,
    },
    "fp16_cuda_matmul": {
        "status": "passed",
        "shape": [257, 193, 131],
        "max_abs_error": max_abs_error,
        "relative_max_error": relative_max_error,
        "checksum": float(product_fp16.float().sum().item()),
    },
    "torchvision_cuda_nms": {
        "status": "passed",
        "keep": nms_keep.cpu().tolist(),
    },
    "vendored_timm": {
        "version": timm.__version__,
        "root": str(VENDORED_TIMM_ROOT),
        "module": str(observed_timm_root),
        "installed_distribution_ignored": distribution_version("timm"),
    },
    "distributions": runtime_distributions,
    "modules": runtime_modules,
    "selected_environment": {
        key: os.environ.get(key)
        for key in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "CUDA_VISIBLE_DEVICES", "TQDM_DISABLE")
    },
    "dependency_install_performed": False,
    "progress": {
        "compact": bool(KAGGLE_COMPACT_PROGRESS),
        "tqdm_disable": os.environ.get("TQDM_DISABLE"),
        "tqdm_probe_disabled": bool(_progress_probe.disable),
        "epoch_metric_logs_retained": True,
    },
}
runtime_fingerprint_payload = json.dumps(
    RUNTIME_CONTRACT,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")
RUNTIME_CONTRACT["compatibility_fingerprint_sha256"] = hashlib.sha256(
    runtime_fingerprint_payload
).hexdigest()
RUNTIME_CONTRACT_PATH = WORK_ROOT / "kaggle_v170_runtime_contract.json"
RUNTIME_CONTRACT_PATH.write_text(
    json.dumps(RUNTIME_CONTRACT, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
del left, right, product_fp16, reference_fp32, nms_boxes, nms_scores, nms_keep
torch.cuda.empty_cache()
print(json.dumps(RUNTIME_CONTRACT, ensure_ascii=False, indent=2))


In [ ]:
import yaml

def ordered_names(document):
    names = document.get("names") if isinstance(document, dict) else None
    if isinstance(names, list):
        return [str(name) for name in names]
    if isinstance(names, dict):
        try:
            keys = sorted(int(key) for key in names)
            if keys != list(range(len(keys))):
                return []
            return [str(names.get(key, names.get(str(key)))) for key in keys]
        except (TypeError, ValueError):
            return []
    return []

def resolve_override(value):
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path
    matches = sorted({(root / path).resolve() for root in SEARCH_ROOTS if (root / path).exists()})
    if len(matches) > 1:
        raise RuntimeError(f"Override mo ho, tim thay nhieu path: {matches}")
    return matches[0] if matches else INPUT_ROOT / path

def data_document_compatible(document):
    if not isinstance(document, dict):
        return False
    try:
        num_classes = int(document.get("nc", 5))
    except (TypeError, ValueError):
        return False
    return bool(
        ordered_names(document) == list(EXPECTED_CLASS_NAMES)
        and document.get("train")
        and document.get("val")
        and num_classes == 5
    )

if DATA_YAML_OVERRIDE:
    data_candidates = [resolve_override(DATA_YAML_OVERRIDE)]
else:
    data_candidates = []
    for path in find_in_search_roots("data.yaml"):
        if REPO_ROOT in path.parents:
            continue
        try:
            candidate_document = yaml.safe_load(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if data_document_compatible(candidate_document):
            data_candidates.append(path)
if len(data_candidates) != 1 or not data_candidates[0].is_file():
    raise RuntimeError(
        "Can dung dung 1 data.yaml classification-folder 5 lop tuong thich; "
        f"tim thay: {data_candidates}. Neu co nhieu file, dat DATA_YAML_OVERRIDE."
    )
DATA_YAML = data_candidates[0].resolve()
DATA_DOCUMENT = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
if not data_document_compatible(DATA_DOCUMENT):
    raise RuntimeError("DATA_YAML_OVERRIDE khong dung schema 5 lop/class order B2")

if any(str(key).casefold() == "test" for key in DATA_DOCUMENT):
    raise RuntimeError("Uploaded data.yaml khong duoc khai bao test")
DATA_FROM_DATASET_ZIP = bool(
    DATASET_EXTRACT_ROOT is not None
    and DATA_YAML.is_relative_to(DATASET_EXTRACT_ROOT.resolve())
)
if DATASET_EXTRACT_ROOT is not None:
    zip_data_yamls = []
    for path in DATASET_EXTRACT_ROOT.rglob("data.yaml"):
        try:
            payload = yaml.safe_load(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if data_document_compatible(payload):
            zip_data_yamls.append(path.resolve())
    if len(zip_data_yamls) != 1:
        raise RuntimeError(f"Dataset ZIP phai co dung 1 compatible data.yaml: {zip_data_yamls}")
    zip_data_document = yaml.safe_load(zip_data_yamls[0].read_text(encoding="utf-8"))
    if any(str(key).casefold() == "test" for key in zip_data_document):
        raise RuntimeError("Dataset ZIP data.yaml khong duoc khai bao test")
    if not DATA_FROM_DATASET_ZIP or DATA_YAML != zip_data_yamls[0]:
        raise RuntimeError("Khi attach dataset ZIP, DATA_YAML phai nam trong ZIP do")

DATASET_INPUT_ANCHOR = DATASET_ZIP if DATASET_ZIP is not None else DATA_YAML
DATASET_INPUT_MOUNT = kaggle_input_mount(DATASET_INPUT_ANCHOR)
if DATASET_INPUT_MOUNT == ASSET_INPUT_MOUNT:
    raise RuntimeError(
        "Asset va dataset phai la hai Kaggle Dataset inputs rieng; "
        f"khong gop chung trong {ASSET_INPUT_MOUNT}"
    )
if DATASET_ZIP is None:
    invalid_dataset_entries = [
        str(path)
        for path in DATASET_INPUT_MOUNT.rglob("*")
        if path.is_symlink() or not (path.is_file() or path.is_dir())
    ]
    if invalid_dataset_entries:
        raise RuntimeError(
            f"Expanded dataset chua symlink/special entries: {invalid_dataset_entries[:20]}"
        )
    forbidden_test_paths = [
        str(path)
        for path in DATASET_INPUT_MOUNT.rglob("*")
        if any(part.casefold() == "test" for part in path.relative_to(DATASET_INPUT_MOUNT).parts)
    ]
    if forbidden_test_paths:
        raise RuntimeError(
            f"Expanded dataset khong duoc chua test component: {forbidden_test_paths[:20]}"
        )
    DATASET_ZIP_CONTRACT = {
        "schema_version": 1,
        "status": "passed",
        "label": "dataset",
        "input_mode": "kaggle_mounted_expanded",
        "root": str(DATASET_INPUT_MOUNT),
        "data_yaml": str(DATA_YAML),
        "data_yaml_sha256": sha256(DATA_YAML),
    }
DATASET_INPUT_CONTRACT = DATASET_ZIP_CONTRACT
DATASET_CONTENT_ROOT = (
    DATASET_EXTRACT_ROOT.resolve()
    if DATASET_EXTRACT_ROOT is not None
    else DATASET_INPUT_MOUNT
)

def split_path_for_root(root, value):
    raw = Path(str(value))
    candidates = [raw] if raw.is_absolute() else [root / raw]
    tail = re.split(r"[\\/]", str(value).rstrip("/\\"))[-1]
    candidates.append(root / tail)
    return next((path.resolve() for path in candidates if path.is_dir()), None)

if DATA_ROOT_OVERRIDE:
    root_candidates = [resolve_override(DATA_ROOT_OVERRIDE)]
else:
    configured_root = str(DATA_DOCUMENT.get("path", "")).strip()
    root_candidates = [DATA_YAML.parent]
    if configured_root:
        raw_root = Path(configured_root)
        root_candidates.append(raw_root if raw_root.is_absolute() else DATA_YAML.parent / raw_root)
        root_tail = re.split(r"[\\/]", configured_root.rstrip("/\\"))[-1]
        root_candidates.extend([DATA_YAML.parent / root_tail, DATA_YAML.parent.parent / root_tail])
valid_roots = []
for root in root_candidates:
    root = root.resolve()
    if split_path_for_root(root, DATA_DOCUMENT["train"]) and split_path_for_root(root, DATA_DOCUMENT["val"]):
        if root not in valid_roots:
            valid_roots.append(root)
if not valid_roots:
    raise RuntimeError("Khong tim thay thu muc train/val. Hay dat DATA_ROOT_OVERRIDE.")
DATA_ROOT = valid_roots[0]
for split_name in ("train", "val"):
    resolved_split = split_path_for_root(DATA_ROOT, DATA_DOCUMENT[split_name])
    if resolved_split is None or not resolved_split.is_relative_to(DATASET_CONTENT_ROOT):
        raise RuntimeError(
            f"{split_name} escapes the selected dataset input: {resolved_split}"
        )

weight_candidates = []
for path in find_in_search_roots("*.safetensors"):
    if path.is_file() and path.stat().st_size == DINO_BYTES and sha256(path) == DINO_SHA256:
        weight_candidates.append(path)
if len(weight_candidates) != 1:
    raise RuntimeError(f"Can dung 1 DINOv3 weight dung hash/kich thuoc, tim thay: {weight_candidates}")
DINO_WEIGHT = weight_candidates[0].resolve()
if ASSET_EXTRACT_ROOT is not None:
    expected_repo_root = (ASSET_EXTRACT_ROOT / "TRKH_pretrained").resolve()
    expected_dino_weight = (ASSET_EXTRACT_ROOT / "weights" / "model.safetensors").resolve()
    if REPO_ROOT.resolve() != expected_repo_root or DINO_WEIGHT != expected_dino_weight:
        raise RuntimeError(
            "Asset input layout/source selection mismatch: "
            f"repo={REPO_ROOT}, dino={DINO_WEIGHT}"
        )

sys.path.insert(0, str(REPO_ROOT))
os.environ["PYTHONPATH"] = (
    str(VENDORED_TIMM_ROOT) + os.pathsep + str(REPO_ROOT)
)
os.environ["MPLBACKEND"] = "Agg"
os.chdir(REPO_ROOT)
if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SOURCE_COMMIT) is None:
    raise RuntimeError("Notebook release chua duoc khoa source commit")
if re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SOURCE_TREE_SHA256) is None:
    raise RuntimeError("Notebook release chua duoc khoa source-tree digest")
source_digest = hashlib.sha256()
source_files = (
    sorted(
        (REPO_ROOT / "trkh").rglob("*.py"),
        key=lambda path: path.relative_to(REPO_ROOT).as_posix(),
    )
    + sorted(
        (REPO_ROOT / "configs").rglob("*.yaml"),
        key=lambda path: path.relative_to(REPO_ROOT).as_posix(),
    )
)
for path in source_files:
    source_digest.update(path.relative_to(REPO_ROOT).as_posix().encode("utf-8") + b"\0")
    source_digest.update(path.read_bytes().replace(b"\r\n", b"\n"))
observed_source_tree_sha256 = source_digest.hexdigest()
if observed_source_tree_sha256 != EXPECTED_SOURCE_TREE_SHA256:
    raise RuntimeError(f"Sai source snapshot: {observed_source_tree_sha256} != {EXPECTED_SOURCE_TREE_SHA256}")
source_commit_file = REPO_ROOT / "SOURCE_COMMIT.txt"
if not source_commit_file.is_file():
    raise RuntimeError("Source archive phai co SOURCE_COMMIT.txt tai repo root")
SOURCE_COMMIT = source_commit_file.read_text(encoding="utf-8").strip().lower()
if SOURCE_COMMIT != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError(f"Sai source commit: {SOURCE_COMMIT} != {EXPECTED_SOURCE_COMMIT}")
SOURCE_TREE_SHA256 = EXPECTED_SOURCE_TREE_SHA256
print("asset_input =", ASSET_INPUT_CONTRACT, "root =", ASSET_EXTRACT_ROOT)
print("dataset_input =", DATASET_INPUT_CONTRACT, "content_root =", DATASET_CONTENT_ROOT)
print("repo =", REPO_ROOT)
print("source_commit =", SOURCE_COMMIT, "source_tree_sha256 =", SOURCE_TREE_SHA256)
print("data =", DATA_YAML)
print("dino =", DINO_WEIGHT)


In [ ]:
from PIL import Image
from trkh.core.config import load_data_spec
from trkh.data.dataset import ClassificationFolderDataset
from trkh.recipes.pretrained_classf_b0 import (
    DINO_SHA256 as RECIPE_DINO_SHA256,
    EXPECTED_CLASS_NAMES as RECIPE_CLASS_NAMES,
    B2_TEMPERED_P05_PROTOCOL_ID as RECIPE_PROTOCOL_ID,
)

if (
    tuple(EXPECTED_CLASS_NAMES) != tuple(RECIPE_CLASS_NAMES)
    or PROTOCOL_ID != RECIPE_PROTOCOL_ID
    or DINO_SHA256 != RECIPE_DINO_SHA256
):
    raise RuntimeError("Notebook drifted from the locked local B2 source recipe")
TRAIN_DIR = split_path_for_root(DATA_ROOT, DATA_DOCUMENT["train"])
VAL_DIR = split_path_for_root(DATA_ROOT, DATA_DOCUMENT["val"])
if TRAIN_DIR is None or VAL_DIR is None:
    raise RuntimeError("Khong resolve duoc train/val cua dataset upload")
document = dict(DATA_DOCUMENT)
document["path"] = str(DATA_ROOT)
document["train"] = str(TRAIN_DIR)
document["val"] = str(VAL_DIR)
document.pop("test", None)
DEV_YAML = WORK_ROOT / "class_f_dev_test_locked.yaml"
DEV_YAML.write_text(yaml.safe_dump(document, sort_keys=False, allow_unicode=True), encoding="utf-8")
dev_spec = load_data_spec(DEV_YAML, class_name_mode="raw", expected_num_classes=5)
if dev_spec.data_format != "classification_folder":
    raise RuntimeError(f"Can classification_folder, nhan {dev_spec.data_format}")
if dev_spec.has_test_split:
    raise RuntimeError("Test lock failed: development YAML van co test")
if tuple(dev_spec.class_names) != tuple(EXPECTED_CLASS_NAMES):
    raise RuntimeError(f"Sai class order: {dev_spec.class_names}")

split_datasets = {
    split: ClassificationFolderDataset.from_data_spec(dev_spec, split=split)
    for split in ("train", "val")
}
split_counts = {split: dataset.class_counts(5) for split, dataset in split_datasets.items()}
if any(count <= 0 for counts in split_counts.values() for count in counts):
    raise RuntimeError(f"Moi lop phai co mau trong train va val: {split_counts}")

def fingerprint_split(split, dataset):
    digest = hashlib.sha256()
    content_hashes = set()
    for sample in sorted(dataset.samples, key=lambda item: str(item.image_path).lower()):
        path = sample.image_path.resolve()
        relative = path.relative_to(dataset.root_dir.resolve()).as_posix()
        file_sha = sha256(path)
        with Image.open(path) as image:
            image.verify()
        digest.update(f"{split}\0{int(sample.label)}\0{relative}\0{file_sha}\n".encode("utf-8"))
        content_hashes.add(file_sha)
    return digest.hexdigest(), content_hashes

split_sha256 = {}
split_content_hashes = {}
for split, dataset in split_datasets.items():
    split_sha256[split], split_content_hashes[split] = fingerprint_split(split, dataset)
cross_split_duplicates = split_content_hashes["train"] & split_content_hashes["val"]
if cross_split_duplicates:
    raise RuntimeError(f"Phat hien {len(cross_split_duplicates)} anh trung byte giua train/val")
development_tree_digest = hashlib.sha256()
for split in ("train", "val"):
    development_tree_digest.update(f"{split}\0{split_sha256[split]}\n".encode("utf-8"))
DEVELOPMENT_IMAGE_TREE_SHA256 = development_tree_digest.hexdigest()
dataset_contract = {
    "schema_version": 1,
    "contract": "TRKH_KAGGLE_CLASSF_COMPATIBLE_5CLASS_V1",
    "status": "passed",
    "input_mode": DATASET_INPUT_CONTRACT["input_mode"],
    "dataset_input": DATASET_INPUT_CONTRACT,
    "uploaded_data_yaml": str(DATA_YAML),
    "uploaded_data_yaml_sha256": sha256(DATA_YAML),
    "development_data_yaml": str(DEV_YAML),
    "development_data_yaml_sha256": sha256(DEV_YAML),
    "root": str(DATA_ROOT),
    "class_names": list(dev_spec.class_names),
    "split_class_counts": split_counts,
    "split_totals": {split: len(dataset) for split, dataset in split_datasets.items()},
    "split_sha256": split_sha256,
    "development_image_tree_sha256": DEVELOPMENT_IMAGE_TREE_SHA256,
    "development_image_file_count": sum(len(dataset) for dataset in split_datasets.values()),
    "cross_split_exact_duplicate_count": 0,
    "all_train_val_images_decodable": True,
    "test_key_observed_in_uploaded_yaml": "test" in DATA_DOCUMENT,
    "test_split_content_opened_or_hashed": False,
    "test_model_inference_performed": False,
    "test_metrics_read": False,
}
development_contract = dataset_contract
DATASET_CONTRACT_PATH = WORK_ROOT / "kaggle_development_dataset_contract.json"
DATASET_CONTRACT_PATH.write_text(json.dumps(dataset_contract, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Dataset-compatible contract passed; test is absent from development commands:", DEV_YAML)
print(json.dumps(dataset_contract, ensure_ascii=False, indent=2))


In [ ]:
import torch
from trkh.recipes.pretrained_classf_b0 import (
    _best_history_row,
    _preflight_model,
    build_train_args,
    run_name,
    validate_auto_resume_checkpoint,
)

if RUNTIME_CONTRACT.get("status") != "passed":
    raise RuntimeError("Kaggle runtime contract was not established")
properties = torch.cuda.get_device_properties(0)
vram_gib = properties.total_memory / (1024 ** 3)
MICRO_BATCH = 24 if vram_gib >= 14.5 else 16
GRAD_ACCUM = {24: 2, 16: 3}[MICRO_BATCH]
AMP_DTYPE = "fp16"
AMP_INIT_SCALE = 1024.0
EVAL_BATCH = 64 if vram_gib >= 14.5 else 32
NUM_WORKERS = 2
EVAL_NUM_WORKERS = 2
os.environ["TRKH_AMP_DTYPE"] = AMP_DTYPE
os.environ.setdefault("OMP_NUM_THREADS", "4")

PROBE_TAG = f"{RUN_TAG}_probe"
FULL_TAG = f"{RUN_TAG}_full"

def stage_train_args(stage, run_tag, auto_resume=False):
    return build_train_args(
        data_yaml=DEV_YAML,
        dino_checkpoint=DINO_WEIGHT,
        output_dir=RUNS_ROOT,
        stage=stage,
        run_tag=run_tag,
        batch_size=MICRO_BATCH,
        num_workers=NUM_WORKERS,
        eval_num_workers=EVAL_NUM_WORKERS,
        seed=42,
        amp_init_scale=AMP_INIT_SCALE,
        auto_resume=auto_resume,
        source_commit=SOURCE_COMMIT,
        source_tree_sha256=SOURCE_TREE_SHA256,
        dataset_image_tree_sha256=DEVELOPMENT_IMAGE_TREE_SHA256,
        experiment=EXPERIMENT_KEY,
    )

def training_command(stage, run_tag, auto_resume=False):
    return [sys.executable, "-m", "trkh.training.train", *stage_train_args(stage, run_tag, auto_resume)]

print(properties.name, f"{vram_gib:.1f} GiB", AMP_DTYPE, f"batch={MICRO_BATCH}x{GRAD_ACCUM}")
print("probe:", shlex.join(training_command("probe", PROBE_TAG)))
print("full:", shlex.join(training_command("full", FULL_TAG, AUTO_RESUME)))


In [ ]:
def run_checked(arguments, required=True):
    command = [str(item) for item in arguments]
    print("\n$", shlex.join(command), flush=True)
    child_env = os.environ.copy()
    if KAGGLE_COMPACT_PROGRESS:
        child_env["TQDM_DISABLE"] = "1"
    result = subprocess.run(command, cwd=REPO_ROOT, env=child_env, check=False)
    if required and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    return result.returncode

focused_tests = [
    REPO_ROOT / "tests" / "test_pretrained_semantic_branch.py",
    REPO_ROOT / "tests" / "test_timm_classifier_model.py",
    REPO_ROOT / "tests" / "test_canonical_classf_defaults.py",
    REPO_ROOT / "tests" / "test_deploy_classification_folder.py",
    REPO_ROOT / "tests" / "test_pretrained_classf_recipe.py",
    REPO_ROOT / "tests" / "test_resume_weight_and_distillation_source.py",
    REPO_ROOT / "tests" / "test_attention_viz_headless.py",
]
if "FOCUSED_TEST_RESULT" not in globals():
    test_returncode = run_checked([
        sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *focused_tests
    ])
    FOCUSED_TEST_RESULT = {"returncode": test_returncode, "paths": [str(path) for path in focused_tests]}

def run_stage(stage, run_tag, auto_resume=False):
    train_args = stage_train_args(stage, run_tag, auto_resume)
    run_dir = RUNS_ROOT / run_name(stage, run_tag, experiment=EXPERIMENT_KEY)
    preflight_dir = RUNS_ROOT / f"preflight_classf_b2_tempered_p05_{run_tag}"
    preflight_dir.mkdir(parents=True, exist_ok=True)
    preflight_path = preflight_dir / f"{stage}_manifest.json"
    if run_dir.exists() and any(run_dir.iterdir()) and not auto_resume:
        raise RuntimeError(f"Run da ton tai: {run_dir}; doi RUN_TAG hoac dung AUTO_RESUME cho full")
    resume_contract = None
    if auto_resume:
        last_checkpoint = run_dir / "checkpoints" / "last.pt"
        if not last_checkpoint.is_file():
            raise FileNotFoundError(f"AUTO_RESUME requires {last_checkpoint}")
        resume_contract = validate_auto_resume_checkpoint(
            last_checkpoint,
            training_data_yaml=DEV_YAML,
            expected_train_args=train_args,
        )
    model_preflight = _preflight_model(train_args)
    preflight = {
        "schema_version": 1,
        "protocol": PROTOCOL_ID,
        "experiment": EXPERIMENT_KEY,
        "adapter": "KAGGLE_CLASSF_COMPATIBLE_DATA_V1",
        "kaggle_notebook_contract": KAGGLE_NOTEBOOK_CONTRACT,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "test_locked": True,
        "test_split_content_opened_or_hashed": False,
        "old_dataset_checkpoint_or_teacher_used": False,
        "dataset": dataset_contract,
        "source_commit": SOURCE_COMMIT,
        "source_tree_sha256": SOURCE_TREE_SHA256,
        "dino_sha256": DINO_SHA256,
        "upload_inputs": {
            "asset_input": ASSET_INPUT_CONTRACT,
            "asset_manifest_sha256": sha256(manifest_path) if manifest_path is not None else None,
            "dataset_input": DATASET_INPUT_CONTRACT,
        },
        "focused_tests": FOCUSED_TEST_RESULT,
        "model_preflight": model_preflight,
        "runtime_contract": RUNTIME_CONTRACT,
        "runtime_contract_path": str(RUNTIME_CONTRACT_PATH),
        "smoke_reload_contract": globals().get("SMOKE_RELOAD_CONTRACT"),
        "runtime": {
            "python": sys.version,
            "torch": torch.__version__,
            "gpu": properties.name,
            "vram_gib": vram_gib,
            "amp_dtype": AMP_DTYPE,
            "micro_batch": MICRO_BATCH,
            "grad_accum": GRAD_ACCUM,
            "effective_batch": MICRO_BATCH * GRAD_ACCUM,
            "amp_init_scale": AMP_INIT_SCALE,
            "num_workers": NUM_WORKERS,
            "eval_num_workers": EVAL_NUM_WORKERS,
        },
        "resume_contract": resume_contract,
        "train_args": train_args,
    }
    preflight_path.write_text(json.dumps(preflight, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    returncode = run_checked([sys.executable, "-m", "trkh.training.train", *train_args], required=False)
    marker = {
        "schema_version": 1,
        "protocol": PROTOCOL_ID,
        "experiment": EXPERIMENT_KEY,
        "adapter": "KAGGLE_CLASSF_COMPATIBLE_DATA_V1",
        "stage": stage,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "returncode": returncode,
        "preflight_manifest": str(preflight_path),
        "dataset_image_tree_sha256": DEVELOPMENT_IMAGE_TREE_SHA256,
        "source_tree_sha256": SOURCE_TREE_SHA256,
        "dino_sha256": DINO_SHA256,
        "test_locked": True,
        "metrics": _best_history_row(run_dir),
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    marker_path = run_dir / "b2_stage_complete.json"
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    if returncode != 0:
        raise RuntimeError(
            f"B2 {stage} failed: {marker_path}. "
            "If Kaggle reports a DataLoader worker-start failure, set "
            "NUM_WORKERS=0 and EVAL_NUM_WORKERS=0 before retrying with a new RUN_TAG; "
            "the notebook will not auto-retry a possibly partial run."
        )
    return run_dir, preflight_path, marker_path

SMOKE_TAG = f"{RUN_TAG}_smoke"
PROBE_TAG = f"{RUN_TAG}_probe"
FULL_TAG = f"{RUN_TAG}_full"

def validate_completed_stage(stage, run_tag, marker_path, auto_resume=False):
    marker = json.loads(marker_path.read_text(encoding="utf-8"))
    lineage = (
        marker.get("dataset_image_tree_sha256"),
        marker.get("source_tree_sha256"),
        marker.get("dino_sha256"),
    )
    expected_lineage = (DEVELOPMENT_IMAGE_TREE_SHA256, SOURCE_TREE_SHA256, DINO_SHA256)
    if lineage != expected_lineage:
        raise RuntimeError(f"{stage} marker lineage mismatch: {lineage}")
    preflight_path = Path(str(marker.get("preflight_manifest", "")))
    if not preflight_path.is_file():
        raise RuntimeError(f"{stage} marker is missing its preflight manifest")
    preflight = json.loads(preflight_path.read_text(encoding="utf-8"))
    if preflight.get("train_args") != stage_train_args(stage, run_tag, auto_resume):
        raise RuntimeError(f"{stage} marker train contract mismatch; use a new RUN_TAG")
    if marker.get("returncode") != 0:
        raise RuntimeError(f"{stage} marker records a failed process: {marker}")
    return marker, preflight_path

SMOKE_DIR = RUNS_ROOT / run_name("smoke", SMOKE_TAG, experiment=EXPERIMENT_KEY)
smoke_train_args = stage_train_args("smoke", SMOKE_TAG)
def train_argument_value(arguments, flag):
    try:
        return arguments[arguments.index(flag) + 1]
    except (ValueError, IndexError) as error:
        raise RuntimeError(f"Training contract is missing {flag}") from error

smoke_batch_contract = {
    "max_train_batches": train_argument_value(smoke_train_args, "--max-train-batches"),
    "max_val_batches": train_argument_value(smoke_train_args, "--max-val-batches"),
    "epochs": train_argument_value(smoke_train_args, "--epochs"),
}
if smoke_batch_contract != {
    "max_train_batches": "4",
    "max_val_batches": "2",
    "epochs": "1",
}:
    raise RuntimeError(f"Smoke must be exactly 4 train / 2 val batches: {smoke_batch_contract}")
smoke_marker_path = SMOKE_DIR / "b2_stage_complete.json"
if not smoke_marker_path.is_file():
    _, _, smoke_marker_path = run_stage("smoke", SMOKE_TAG)
smoke_marker, SMOKE_PREFLIGHT_MANIFEST = validate_completed_stage(
    "smoke", SMOKE_TAG, smoke_marker_path
)
SMOKE_CHECKPOINT = SMOKE_DIR / "checkpoints" / "best.pt"
if not SMOKE_CHECKPOINT.is_file():
    SMOKE_CHECKPOINT = SMOKE_DIR / "checkpoints" / "last.pt"
if not SMOKE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Smoke stage did not produce a checkpoint: {SMOKE_DIR}")

# A successful process/checkpoint reload is insufficient when GradScaler skipped
# every optimizer step. Require explicit epoch telemetry and non-empty AdamW state.
SMOKE_HISTORY = SMOKE_DIR / "history.csv"
if not SMOKE_HISTORY.is_file():
    raise FileNotFoundError(f"Smoke stage did not write history.csv: {SMOKE_DIR}")
with SMOKE_HISTORY.open("r", encoding="utf-8-sig", newline="") as handle:
    smoke_history_rows = list(csv.DictReader(handle))
if len(smoke_history_rows) != 1:
    raise RuntimeError(f"Smoke must write exactly one epoch row, observed {len(smoke_history_rows)}")
smoke_history_row = smoke_history_rows[0]
smoke_optimizer_telemetry = {
    key: float(smoke_history_row.get(key, "nan"))
    for key in (
        "train_optimizer_step_attempts",
        "train_optimizer_updates_successful",
        "train_optimizer_steps_skipped_nonfinite",
        "train_nonfinite_loss_batches",
        "train_amp_optimizer_steps_skipped",
    )
}
if not all(math.isfinite(value) for value in smoke_optimizer_telemetry.values()):
    raise RuntimeError(f"Smoke optimizer telemetry is missing/non-finite: {smoke_optimizer_telemetry}")
if smoke_optimizer_telemetry["train_optimizer_step_attempts"] < 1:
    raise RuntimeError(f"Smoke made no optimizer attempt: {smoke_optimizer_telemetry}")
if smoke_optimizer_telemetry["train_optimizer_updates_successful"] < 1:
    raise RuntimeError(f"Smoke made no real optimizer update: {smoke_optimizer_telemetry}")
for key in (
    "train_optimizer_steps_skipped_nonfinite",
    "train_nonfinite_loss_batches",
    "train_amp_optimizer_steps_skipped",
):
    if smoke_optimizer_telemetry[key] != 0:
        raise RuntimeError(f"Smoke numerical-stability gate failed: {smoke_optimizer_telemetry}")

SMOKE_STATE_CHECKPOINT = SMOKE_DIR / "checkpoints" / "last.pt"
if not SMOKE_STATE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Smoke stage did not write last.pt state checkpoint: {SMOKE_DIR}")
smoke_checkpoint_payload = torch.load(
    SMOKE_STATE_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)
smoke_optimizer_state_entries = len(
    smoke_checkpoint_payload.get("optimizer_state", {}).get("state", {})
)
smoke_scaler_state = smoke_checkpoint_payload.get("scaler_state", {})
smoke_checkpoint_scale = float(smoke_scaler_state.get("scale", float("nan")))
del smoke_checkpoint_payload
if smoke_optimizer_state_entries < 1:
    raise RuntimeError("Smoke checkpoint has empty optimizer state despite a successful process")
if not math.isfinite(smoke_checkpoint_scale) or smoke_checkpoint_scale != AMP_INIT_SCALE:
    raise RuntimeError(
        "Smoke GradScaler drift/overflow: "
        f"checkpoint_scale={smoke_checkpoint_scale}, expected={AMP_INIT_SCALE}"
    )
SMOKE_OPTIMIZER_CONTRACT = {
    "status": "passed",
    "history": str(SMOKE_HISTORY),
    "history_sha256": sha256(SMOKE_HISTORY),
    "state_checkpoint": str(SMOKE_STATE_CHECKPOINT),
    "state_checkpoint_sha256": sha256(SMOKE_STATE_CHECKPOINT),
    "amp_init_scale": AMP_INIT_SCALE,
    "checkpoint_scale": smoke_checkpoint_scale,
    "optimizer_state_entries": smoke_optimizer_state_entries,
    **smoke_optimizer_telemetry,
}

# A separate evaluator process must reload the checkpoint and complete one val batch.
smoke_checkpoint_sha256 = sha256(SMOKE_CHECKPOINT)
SMOKE_RELOAD_DIR = SMOKE_DIR / f"reload_eval_1batch_{smoke_checkpoint_sha256[:12]}"
SMOKE_RELOAD_MARKER_PATH = SMOKE_RELOAD_DIR / "checkpoint_reload_contract.json"
if not SMOKE_RELOAD_MARKER_PATH.is_file():
    run_checked([
        sys.executable, "-m", "trkh.evaluation.evaluate",
        "--checkpoint", SMOKE_CHECKPOINT,
        "--data", DEV_YAML,
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--split", "val",
        "--batch-size", str(EVAL_BATCH),
        "--num-workers", "2",
        "--amp",
        "--max-batches", "1",
        "--paper-name", "TRKH-DINOv3-ClassF-B2-Kaggle-Smoke-Reload",
        "--family", "TRKH-Pretrained-Engineering-Smoke",
        "--output-dir", SMOKE_RELOAD_DIR,
    ])
    smoke_predictions = SMOKE_RELOAD_DIR / "predictions_detailed.csv"
    if not smoke_predictions.is_file():
        smoke_predictions = SMOKE_RELOAD_DIR / "predictions.csv"
    smoke_metrics = SMOKE_RELOAD_DIR / "metrics_detailed.json"
    if not smoke_metrics.is_file():
        smoke_metrics = SMOKE_RELOAD_DIR / "metrics.json"
    if not smoke_predictions.is_file() or not smoke_metrics.is_file():
        raise FileNotFoundError("Independent smoke checkpoint reload did not write metrics/predictions")
    with smoke_predictions.open("r", encoding="utf-8-sig", newline="") as handle:
        smoke_prediction_rows = sum(1 for _ in csv.DictReader(handle))
    if smoke_prediction_rows < 1 or smoke_prediction_rows > EVAL_BATCH:
        raise RuntimeError(f"Unexpected one-batch prediction rows: {smoke_prediction_rows}")
    SMOKE_RELOAD_CONTRACT = {
        "schema_version": 1,
        "status": "passed",
        "separate_process_checkpoint_reload": True,
        "max_val_batches": 1,
        "checkpoint": str(SMOKE_CHECKPOINT),
        "checkpoint_sha256": smoke_checkpoint_sha256,
        "metrics": str(smoke_metrics),
        "metrics_sha256": sha256(smoke_metrics),
        "predictions": str(smoke_predictions),
        "predictions_sha256": sha256(smoke_predictions),
        "prediction_rows": smoke_prediction_rows,
        "optimizer_update_contract": SMOKE_OPTIMIZER_CONTRACT,
        "runtime_fingerprint_sha256": RUNTIME_CONTRACT["compatibility_fingerprint_sha256"],
    }
    SMOKE_RELOAD_DIR.mkdir(parents=True, exist_ok=True)
    SMOKE_RELOAD_MARKER_PATH.write_text(
        json.dumps(SMOKE_RELOAD_CONTRACT, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
else:
    SMOKE_RELOAD_CONTRACT = json.loads(SMOKE_RELOAD_MARKER_PATH.read_text(encoding="utf-8"))
    expected_reload = {
        "status": "passed",
        "separate_process_checkpoint_reload": True,
        "max_val_batches": 1,
        "checkpoint_sha256": smoke_checkpoint_sha256,
        "optimizer_update_contract": SMOKE_OPTIMIZER_CONTRACT,
        "runtime_fingerprint_sha256": RUNTIME_CONTRACT["compatibility_fingerprint_sha256"],
    }
    reload_mismatches = {
        key: {"observed": SMOKE_RELOAD_CONTRACT.get(key), "expected": expected}
        for key, expected in expected_reload.items()
        if SMOKE_RELOAD_CONTRACT.get(key) != expected
    }
    for artifact_key, digest_key in (("metrics", "metrics_sha256"), ("predictions", "predictions_sha256")):
        artifact_path = Path(str(SMOKE_RELOAD_CONTRACT.get(artifact_key, "")))
        if not artifact_path.is_file() or sha256(artifact_path) != SMOKE_RELOAD_CONTRACT.get(digest_key):
            reload_mismatches[artifact_key] = "missing_or_digest_mismatch"
    if reload_mismatches:
        raise RuntimeError(f"Smoke reload contract mismatch: {reload_mismatches}")
print("Smoke 4-train/2-val plus independent one-batch reload passed:", SMOKE_RELOAD_MARKER_PATH)

PROBE_DIR = RUNS_ROOT / run_name("probe", PROBE_TAG, experiment=EXPERIMENT_KEY)
PROBE_COMPLETED = False
if RUN_PROBE:
    probe_marker_path = PROBE_DIR / "b2_stage_complete.json"
    if not probe_marker_path.is_file():
        _, _, probe_marker_path = run_stage("probe", PROBE_TAG)
    probe_marker, PROBE_PREFLIGHT_MANIFEST = validate_completed_stage(
        "probe", PROBE_TAG, probe_marker_path
    )
    probe_metrics = probe_marker.get("metrics", {})
    if float(probe_metrics.get("best_macro_f1") or 0) < PROBE_MIN_MACRO_F1:
        raise RuntimeError(f"Probe macro-F1 gate failed: {probe_metrics}")
    if float(probe_metrics.get("best_class1_f1") or 0) < PROBE_MIN_CLASS1_F1:
        raise RuntimeError(f"Probe class-1 gate failed: {probe_metrics}")
    PROBE_COMPLETED = True
    print("Probe gate passed:", probe_metrics)
else:
    print("Probe execution skipped only because AUTO_RESUME=True; resume validation remains fail-closed.")

FULL_REQUESTED = bool(CONFIRM_FULL or AUTO_RESUME)
FULL_COMPLETED = False
RUN_DIR = None
FULL_PREFLIGHT_MANIFEST = None
BEST_CHECKPOINT = None
if FULL_REQUESTED:
    if not (PROBE_COMPLETED or AUTO_RESUME):
        raise RuntimeError("Full requires a completed probe or provenance-checked AUTO_RESUME")
    full_dir = RUNS_ROOT / run_name("full", FULL_TAG, experiment=EXPERIMENT_KEY)
    full_marker_path = full_dir / "b2_stage_complete.json"
    if full_marker_path.is_file():
        _, FULL_PREFLIGHT_MANIFEST = validate_completed_stage(
            "full", FULL_TAG, full_marker_path, AUTO_RESUME
        )
        RUN_DIR = full_dir
    else:
        RUN_DIR, FULL_PREFLIGHT_MANIFEST, full_marker_path = run_stage(
            "full", FULL_TAG, AUTO_RESUME
        )
    BEST_CHECKPOINT = RUN_DIR / "checkpoints" / "best.pt"
    if not BEST_CHECKPOINT.is_file() or not FULL_PREFLIGHT_MANIFEST.is_file():
        raise FileNotFoundError(f"Missing best/preflight: {BEST_CHECKPOINT}, {FULL_PREFLIGHT_MANIFEST}")
    FULL_COMPLETED = True
    print("Best checkpoint:", BEST_CHECKPOINT)
else:
    print(
        "Probe is complete. No exception is raised and no audit is attempted. "
        "Review probe evidence, set CONFIRM_FULL=True, then rerun from the configuration cell."
    )


In [ ]:
if not FULL_COMPLETED:
    EVAL_DIR = PREDICTIONS = METRICS_DETAIL = None
    print("SKIP validation evaluation: full train has not completed.")
else:
    EVAL_DIR = RUN_DIR / "eval_val_full"
    run_checked([
        sys.executable, "-m", "trkh.evaluation.evaluate",
        "--checkpoint", BEST_CHECKPOINT,
        "--data", DEV_YAML,
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--split", "val",
        "--batch-size", EVAL_BATCH,
        "--num-workers", "2",
        "--amp",
        "--max-batches", "0",
        "--paper-name", "TRKH-DINOv3-ClassF-B2-Tempered-P05",
        "--family", "TRKH-Pretrained",
        "--output-dir", EVAL_DIR,
    ])
    PREDICTIONS = EVAL_DIR / "predictions_detailed.csv"
    if not PREDICTIONS.is_file():
        PREDICTIONS = EVAL_DIR / "predictions.csv"
    if not PREDICTIONS.is_file():
        raise FileNotFoundError("Evaluate did not write predictions CSV")
    METRICS_DETAIL = EVAL_DIR / "metrics_detailed.json"
    if not METRICS_DETAIL.is_file():
        METRICS_DETAIL = EVAL_DIR / "metrics.json"
    print(METRICS_DETAIL.read_text(encoding="utf-8")[:4000])


In [ ]:
if not FULL_COMPLETED:
    print("SKIP class-confusion/forensics audits: full train has not completed.")
else:
    run_checked([
        sys.executable, "-m", "trkh.tools.audit_class_confusions",
        "--predictions", PREDICTIONS,
        "--data", DEV_YAML,
        "--focus-class-index", "1",
        "--top-k-images", "32",
        "--copy-images",
        "--output-dir", RUN_DIR / "audit_class1_confusions",
    ])
    run_checked([
        sys.executable, "-m", "trkh.tools.audit_prediction_forensics",
        "--predictions", PREDICTIONS,
        "--pairs", "0-1,1-2,1-4,2-3",
        "--focus-class-index", "1",
        "--ece-bins", "15",
        "--image-stats-mode", "foreground",
        "--max-image-stats", "512",
        "--image-stats-workers", "2",
        "--top-k-images", "32",
        "--copy-images",
        "--output-dir", RUN_DIR / "audit_prediction_forensics",
    ])


In [ ]:
if not FULL_COMPLETED:
    print("SKIP XAI/robustness audits: full train has not completed.")
else:
    run_checked([
        sys.executable, "-m", "trkh.evaluation.xai_audit",
        "--checkpoint", BEST_CHECKPOINT,
        "--data", DEV_YAML,
        "--split", "val",
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--output-dir", RUN_DIR / "xai_val",
        "--max-cases", "24",
        "--mistake-cases", "8",
        "--low-confidence-cases", "4",
        "--close-margin-cases", "4",
        "--per-class-cases", "1",
        "--focus-class-index", "1",
        "--focus-false-positive-cases", "4",
        "--focus-false-negative-cases", "4",
        "--batch-size", EVAL_BATCH,
        "--num-workers", "2",
        "--method", "gradcam",
        "--feature-source", "patch_embed",
        "--robustness-probes",
    ])
    run_checked([
        sys.executable, "-m", "trkh.evaluation.robustness_eval",
        "--checkpoint", BEST_CHECKPOINT,
        "--data", DEV_YAML,
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--batch-size", EVAL_BATCH,
        "--num-workers", "2",
        "--max-batches", "0",
        "--num-fail-cases", "16",
        "--output-dir", RUN_DIR / "robustness_val",
    ])


In [ ]:
deploy_code = None
if not FULL_COMPLETED:
    print("SKIP architecture trace/optional export: full train has not completed.")
else:
    run_checked([
        sys.executable, "-m", "trkh.tools.trace_architecture",
        "--checkpoint", BEST_CHECKPOINT,
        "--data", DEV_YAML,
        "--class-name-mode", "raw",
        "--expected-num-classes", "5",
        "--image-size", "256",
        "--device", "cuda",
        "--seed", "42",
        "--output-dir", RUN_DIR / "architecture_trace",
    ])

    # DINOv3 B2 is a research teacher, not the mobile deliverable.
    deploy_code = None
    if importlib.util.find_spec("onnx") is None:
        print("SKIP optional ONNX export: onnx is absent from the immutable Kaggle runtime.")
    else:
        deploy_code = run_checked([
            sys.executable, "-m", "trkh.inference.deploy",
            "--checkpoint", BEST_CHECKPOINT,
            "--data", DEV_YAML,
            "--class-name-mode", "raw",
            "--expected-num-classes", "5",
            "--output-dir", RUN_DIR / "deploy_teacher_fp32",
            "--batch-size", EVAL_BATCH,
            "--num-workers", "2",
            "--benchmark-batch-size", "1",
            "--benchmark-warmup", "2",
            "--benchmark-runs", "2",
            "--split", "val",
            "--skip-accuracy",
            "--skip-benchmark",
            "--skip-trt-engine",
        ], required=False)
        if deploy_code:
            print("WARNING: optional ONNX export failed; checkpoint/audits remain valid:", deploy_code)


In [ ]:
if not FULL_COMPLETED:
    readiness_manifest = {
        "schema_version": 2,
        "status": "probe_ready_full_not_requested",
        "protocol": PROTOCOL_ID,
        "experiment": EXPERIMENT_KEY,
        "kaggle_notebook_contract": KAGGLE_NOTEBOOK_CONTRACT,
        "runtime_contract": str(RUNTIME_CONTRACT_PATH),
        "runtime_fingerprint_sha256": RUNTIME_CONTRACT["compatibility_fingerprint_sha256"],
        "smoke_stage_marker": str(SMOKE_DIR / "b2_stage_complete.json"),
        "smoke_checkpoint_reload_contract": str(SMOKE_RELOAD_MARKER_PATH),
        "probe_completed": PROBE_COMPLETED,
        "probe_stage_marker": str(PROBE_DIR / "b2_stage_complete.json") if PROBE_COMPLETED else None,
        "full_requested": FULL_REQUESTED,
        "full_completed": False,
        "audits_run": False,
        "next_action": "Review probe, set CONFIRM_FULL=True, rerun from cell 0.",
        "test_split_content_opened_or_hashed": False,
        "test_metrics_read": False,
    }
    SESSION_MANIFEST = WORK_ROOT / f"TRKH_CLASSF_B2_{RUN_TAG}_PROBE_READINESS.json"
    SESSION_MANIFEST.write_text(
        json.dumps(readiness_manifest, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Full was not requested; audit/package cells skipped cleanly.")
    print("Readiness manifest:", SESSION_MANIFEST)
else:
    full_preflight = json.loads(FULL_PREFLIGHT_MANIFEST.read_text(encoding="utf-8"))
    validation_metrics = json.loads(METRICS_DETAIL.read_text(encoding="utf-8"))
    validation_support = sum(int(row["support"]) for row in validation_metrics["per_class"])
    with PREDICTIONS.open("r", encoding="utf-8-sig", newline="") as handle:
        prediction_rows = sum(1 for _ in csv.DictReader(handle))
    expected_validation_support = int(dataset_contract["split_totals"]["val"])
    if validation_support != expected_validation_support or prediction_rows != expected_validation_support:
        raise RuntimeError(
            "Full validation incomplete: "
            f"expected={expected_validation_support}, support={validation_support}, predictions={prediction_rows}"
        )
    class1_metrics = validation_metrics["per_class"][1]
    audit_artifacts = {
        "validation": METRICS_DETAIL,
        "predictions": PREDICTIONS,
        "class1_confusions": RUN_DIR / "audit_class1_confusions",
        "prediction_forensics": RUN_DIR / "audit_prediction_forensics",
        "xai": RUN_DIR / "xai_val" / "xai_audit_summary.json",
        "robustness": RUN_DIR / "robustness_val" / "robustness_summary.json",
        "architecture_trace": RUN_DIR / "architecture_trace" / "trace_summary.json",
    }
    if deploy_code == 0:
        audit_artifacts["technical_onnx_export"] = RUN_DIR / "deploy_teacher_fp32"
    audit_status = {
        name: (path.is_file() or (path.is_dir() and any(path.rglob("*"))))
        for name, path in audit_artifacts.items()
    }
    if not all(audit_status.values()):
        raise RuntimeError(f"Required audit missing: {audit_status}")
    shutil.copy2(DATA_YAML, RUN_DIR / "uploaded_data.yaml")
    shutil.copy2(DEV_YAML, RUN_DIR / DEV_YAML.name)
    shutil.copy2(DATASET_CONTRACT_PATH, RUN_DIR / DATASET_CONTRACT_PATH.name)
    if manifest_path is not None:
        shutil.copy2(manifest_path, RUN_DIR / ASSET_MANIFEST_NAME)
    shutil.copy2(FULL_PREFLIGHT_MANIFEST, RUN_DIR / "b2_full_preflight_manifest.json")
    shutil.copy2(RUNTIME_CONTRACT_PATH, RUN_DIR / RUNTIME_CONTRACT_PATH.name)
    shutil.copy2(SMOKE_RELOAD_MARKER_PATH, RUN_DIR / "smoke_checkpoint_reload_contract.json")
    if (PROBE_DIR / "b2_stage_complete.json").is_file():
        shutil.copy2(PROBE_DIR / "b2_stage_complete.json", RUN_DIR / "b2_probe_stage_complete.json")
    evidence_roots = list(audit_artifacts.values()) + [
        BEST_CHECKPOINT,
        RUN_DIR / "uploaded_data.yaml",
        RUN_DIR / DEV_YAML.name,
        RUN_DIR / DATASET_CONTRACT_PATH.name,
        RUN_DIR / "b2_full_preflight_manifest.json",
        RUN_DIR / RUNTIME_CONTRACT_PATH.name,
        RUN_DIR / "smoke_checkpoint_reload_contract.json",
    ]
    if manifest_path is not None:
        evidence_roots.append(RUN_DIR / ASSET_MANIFEST_NAME)
    evidence_files = set()
    for root in evidence_roots:
        if root.is_file():
            evidence_files.add(root)
        elif root.is_dir():
            evidence_files.update(path for path in root.rglob("*") if path.is_file())
    evidence_inventory = [
        {
            "path": path.relative_to(RUN_DIR).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in sorted(evidence_files)
    ]
    EVIDENCE_INVENTORY = RUN_DIR / "evidence_sha256_inventory.json"
    EVIDENCE_INVENTORY.write_text(json.dumps(evidence_inventory, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    runtime_distributions = {}
    for distribution in ("torch", "torchvision", "timm", "safetensors", "numpy", "Pillow", "PyYAML", "matplotlib", "pytest", "onnx", "onnxruntime"):
        try:
            runtime_distributions[distribution] = version(distribution)
        except PackageNotFoundError:
            runtime_distributions[distribution] = None
    manifest = {
        "schema_version": 1,
        "protocol": PROTOCOL_ID,
        "experiment": EXPERIMENT_KEY,
        "adapter": "KAGGLE_CLASSF_COMPATIBLE_DATA_V1",
        "kaggle_notebook_contract": KAGGLE_NOTEBOOK_CONTRACT,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "uploaded_data_yaml_sha256": dataset_contract["uploaded_data_yaml_sha256"],
        "development_data_yaml_sha256": dataset_contract["development_data_yaml_sha256"],
        "development_image_tree_sha256": DEVELOPMENT_IMAGE_TREE_SHA256,
        "dataset_contract_sha256": sha256(DATASET_CONTRACT_PATH),
        "local_class_f_hash_or_count_compared": False,
        "source_commit": SOURCE_COMMIT,
        "source_tree_sha256": SOURCE_TREE_SHA256,
        "dino_sha256": DINO_SHA256,
        "upload_inputs": {
            "asset_input": ASSET_INPUT_CONTRACT,
            "asset_manifest_sha256": sha256(manifest_path) if manifest_path is not None else None,
            "dataset_input": DATASET_INPUT_CONTRACT,
        },
        "best_checkpoint": str(BEST_CHECKPOINT),
        "best_checkpoint_sha256": sha256(BEST_CHECKPOINT),
        "validation_only": True,
        "validation_support": validation_support,
        "validation_prediction_rows": prediction_rows,
        "validation_metrics": {
            "accuracy": validation_metrics["accuracy"],
            "macro_precision": validation_metrics["macro_precision"],
            "macro_recall": validation_metrics["macro_recall"],
            "macro_f1": validation_metrics["macro_f1"],
            "class1": class1_metrics,
        },
        "audit_status": audit_status,
        "evidence_inventory": str(EVIDENCE_INVENTORY),
        "evidence_inventory_sha256": sha256(EVIDENCE_INVENTORY),
        "evidence_inventory_entries": len(evidence_inventory),
        "test_key_observed_in_uploaded_yaml": dataset_contract["test_key_observed_in_uploaded_yaml"],
        "test_split_content_opened_or_hashed": False,
        "test_pixels_used_by_model": False,
        "test_metrics_or_model_selection_used": False,
        "test_metrics_read": False,
        "old_dataset_checkpoint_or_teacher_used": False,
        "model_recipe_parity": {
            "official_builder": "trkh.recipes.pretrained_classf_b0.build_train_args",
            "experiment": EXPERIMENT_KEY,
            "source_and_dino_locked_to_local_b2": True,
            "data_identity_is_uploaded_run_specific": True,
        },
        "development_data": development_contract,
        "full_preflight_manifest_sha256": sha256(FULL_PREFLIGHT_MANIFEST),
        "train_args": full_preflight["train_args"],
        "python": sys.version,
        "runtime_contract": RUNTIME_CONTRACT,
        "runtime_contract_sha256": sha256(RUN_DIR / RUNTIME_CONTRACT_PATH.name),
        "smoke_checkpoint_reload_contract_sha256": sha256(RUN_DIR / "smoke_checkpoint_reload_contract.json"),
        "runtime_distributions": runtime_distributions,
        "gpu": properties.name,
        "vram_gib": vram_gib,
        "amp_dtype": AMP_DTYPE,
        "micro_batch": MICRO_BATCH,
        "grad_accum": GRAD_ACCUM,
        "deploy_returncode": deploy_code,
    }
    MANIFEST = RUN_DIR / "kaggle_artifact_manifest.json"
    MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    archive_base = WORK_ROOT / f"TRKH_CLASSF_B2_TEMPERED_P05_{RUN_TAG}_RESULTS"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR)
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
    print("Download:", archive_path)

